In [ ]:
!git clone https://github.com/elifsudayi/food-recipe-finder.git
%cd food-recipe-finder
!pip install -q -r requirements.txt
!mkdir -p models/onnx results
!lscpu | grep -E "Model name|^CPU\(s\)"

Cloning into 'food-recipe-finder'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 83 (delta 31), reused 66 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 175.54 KiB | 3.51 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/food-recipe-finder/food-recipe-finder
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz


In [ ]:
import logging
logging.getLogger().setLevel(logging.ERROR)
import sys, json, itertools; sys.path.append(".")
import numpy as np, pandas as pd
from src.retrieval.embed import STEncoder, EmbeddingRetriever, DEFAULT_MODEL
from src.retrieval.tfidf import doc_text, TfidfRetriever
from src.retrieval.metrics import evaluate
from src.quant.onnx_utils import *
from src.utils import hardware_info, model_size_mb

corpus = pd.read_csv("corpus/recipes.csv"); queries = pd.read_csv("corpus/queries.csv")
corpus["description"] = corpus["description"].fillna(""); corpus["diet_tags"] = corpus["diet_tags"].fillna("")

st = STEncoder(device="cpu"); max_len = st.model.max_seq_length
FP32, INT8 = "models/onnx/minilm_fp32.onnx", "models/onnx/minilm_int8.onnx"
export_minilm_onnx(DEFAULT_MODEL, FP32)
quantize_minilm_dynamic(FP32, INT8)
print(f"ONNX fp32 {file_mb(FP32):.1f} MB -> INT8 {file_mb(INT8):.1f} MB")

enc32, enc8 = OnnxEncoder(FP32, DEFAULT_MODEL, max_len), OnnxEncoder(INT8, DEFAULT_MODEL, max_len)
docs = [doc_text(r) for _, r in corpus.iterrows()]
E_st, E_32, E_8 = st.encode(docs), enc32.encode(docs), enc8.encode(docs)
for name, E in [("ONNX fp32", E_32), ("ONNX INT8", E_8)]:
    c = (E * E_st).sum(1)
    print(f"{name} vs PyTorch fp32: cosine avg {c.mean():.5f}, min {c.min():.5f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:955: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX fp32 90.4 MB -> INT8 22.8 MB
ONNX fp32 vs PyTorch fp32: cosine avg 1.00000, min 1.00000
ONNX INT8 vs PyTorch fp32: cosine avg 0.94742, min 0.89737


In [ ]:
systems = {
  "tf-idf stem + constraints":        TfidfRetriever(corpus, "stem", True).rank_ids,
  "MiniLM fp32 PyTorch + constraints": EmbeddingRetriever(corpus, st,    True, doc_emb=E_st).rank_ids,
  "MiniLM fp32 ONNX + constraints":    EmbeddingRetriever(corpus, enc32, True, doc_emb=E_32).rank_ids,
  "MiniLM INT8 ONNX + constraints":    EmbeddingRetriever(corpus, enc8,  True, doc_emb=E_8).rank_ids}
res = {}
for n, fn in systems.items():
    s, by, _ = evaluate(fn, queries); res[n] = {"summary": s, "by_type": by.round(4).to_dict()}
    print(f"{n:36s} MRR {s['MRR']:.3f} | Recall@5 {s['Recall@5']:.3f}")
print(pd.concat({n: pd.DataFrame(v["by_type"])["MRR"] for n, v in res.items()}, axis=1).round(3))

tf-idf stem + constraints            MRR 0.946 | Recall@5 0.855
MiniLM fp32 PyTorch + constraints    MRR 0.938 | Recall@5 0.883
MiniLM fp32 ONNX + constraints       MRR 0.938 | Recall@5 0.883
MiniLM INT8 ONNX + constraints       MRR 0.951 | Recall@5 0.889
            tf-idf stem + constraints  MiniLM fp32 PyTorch + constraints  \
combo                           1.000                              1.000   
diet                            1.000                              1.000   
ingredient                      0.900                              0.803   
semantic                        0.883                              0.950   
time                            1.000                              1.000   

            MiniLM fp32 ONNX + constraints  MiniLM INT8 ONNX + constraints  
combo                                1.000                           1.000  
diet                                 1.000                           1.000  
ingredient                           0.803              

In [ ]:
cyc = itertools.cycle(queries["query"].tolist())
def lat(fn): return time_calls(lambda: fn(next(cyc)), runs=200, warmup=10)

tf = TfidfRetriever(corpus, "stem", True)
r8 = EmbeddingRetriever(corpus, enc8, True, doc_emb=E_8)
L = {"tfidf_search":               lat(lambda q: tf.search(q, 5)),
     "minilm_fp32_pytorch_encode": lat(lambda q: st.encode([q])),
     "minilm_fp32_onnx_encode":    lat(lambda q: enc32.encode([q])),
     "minilm_int8_onnx_encode":    lat(lambda q: enc8.encode([q])),
     "minilm_int8_onnx_search":    lat(lambda q: r8.search(q, 5))}
for k, v in L.items(): print(f"{k:28s} {v['mean_ms']:7.2f} ± {v['std_ms']:.2f} ms")

tfidf_search                    3.41 ± 2.71 ms
minilm_fp32_pytorch_encode     29.51 ± 18.46 ms
minilm_fp32_onnx_encode        10.49 ± 2.52 ms
minilm_int8_onnx_encode         7.93 ± 2.23 ms
minilm_int8_onnx_search         7.03 ± 2.01 ms


In [ ]:
out = {"retrieval": {n: v["summary"] for n, v in res.items()},
       "retrieval_by_type": {n: v["by_type"] for n, v in res.items()},
       "embedding_cosine_vs_pytorch_fp32": {"onnx_fp32_min": float((E_32 * E_st).sum(1).min()),
                                            "onnx_int8_mean": float((E_8 * E_st).sum(1).mean()),
                                            "onnx_int8_min": float((E_8 * E_st).sum(1).min())},
       "size_mb": {"minilm_fp32_pytorch": model_size_mb(st.model), "minilm_fp32_onnx": file_mb(FP32),
                   "minilm_int8_onnx": file_mb(INT8)},
       "latency_cpu_ms": L, "hardware": hardware_info()}
json.dump(out, open("results/quant_minilm_metrics.json", "w"), indent=2)
np.save("models/doc_emb_minilm_int8.npy", E_8)
print(json.dumps(out, indent=2))

{
  "retrieval": {
    "tf-idf stem + constraints": {
      "MRR": 0.9458333333333334,
      "Recall@5": 0.8550000000000001,
      "n_queries": 40
    },
    "MiniLM fp32 PyTorch + constraints": {
      "MRR": 0.9383333333333332,
      "Recall@5": 0.8829166666666666,
      "n_queries": 40
    },
    "MiniLM fp32 ONNX + constraints": {
      "MRR": 0.9383333333333332,
      "Recall@5": 0.8829166666666666,
      "n_queries": 40
    },
    "MiniLM INT8 ONNX + constraints": {
      "MRR": 0.9508333333333333,
      "Recall@5": 0.8891666666666665,
      "n_queries": 40
    }
  },
  "retrieval_by_type": {
    "tf-idf stem + constraints": {
      "MRR": {
        "combo": 1.0,
        "diet": 1.0,
        "ingredient": 0.9,
        "semantic": 0.8833,
        "time": 1.0
      },
      "recall@5": {
        "combo": 1.0,
        "diet": 0.95,
        "ingredient": 0.74,
        "semantic": 0.78,
        "time": 0.925
      }
    },
    "MiniLM fp32 PyTorch + constraints": {
      "MRR": {
    